# 12 · Performance: plans, partitions, cost (Oracle)

Each engine is fast in a different way, so this notebook has **different questions per engine**.

- **BigQuery** charges by **bytes read**. Columnar storage (each column stored apart): read fewer columns = pay less.
  **Partitioning** (split a table by date) and **clustering** (sort inside partitions) let it skip data.
- **Oracle** uses an **execution plan** made by the cost-based optimizer: full table scan, index, join method.
  `EXPLAIN PLAN` shows it.
- **Spark** builds a plan too (`explain()`). Look for **pushed filters**, **partition filters**,
  **broadcast joins** and **exchanges** (shuffles: data moving between machines, the slow part).

The tables here are tiny, so speed differences are small. Learn to **read** the plans: that is what interviews ask.

**How to run:** Start each SQL cell with `%%oracle`. One statement per cell. Tables are in your own schema, so just write `employees`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`oracle/practice/12_performance.ipynb`](../../oracle/practice/12_performance.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "oracle"))
%load_ext oracle_helper

## Q6. Read an execution plan  ·  🟡 medium

Show the plan Oracle picks for: employees joined to departments, with dept_name = 'Sales'. Use `EXPLAIN PLAN FOR ...`, then `DBMS_XPLAN.DISPLAY`.

**How it works:** `EXPLAIN PLAN FOR` saves the plan into PLAN_TABLE; `DBMS_XPLAN.DISPLAY` prints it as text.

In [ ]:
%%oracle
EXPLAIN PLAN FOR
SELECT e.name, d.dept_name
FROM employees e
JOIN departments d ON d.dept_id = e.dept_id
WHERE d.dept_name = 'Sales'

OK

In [ ]:
%%oracle
SELECT plan_table_output FROM TABLE(DBMS_XPLAN.DISPLAY(format => 'BASIC +ROWS +COST'))

,plan_table_output
0,Plan hash value: 2052257371
1,
2,---------------------------------------------------------------
3,| Id | Operation | Name | Rows | Cost (%CPU)|
4,---------------------------------------------------------------
5,| 0 | SELECT STATEMENT | | 5 | 6 (0)|
6,| 1 | HASH JOIN | | 5 | 6 (0)|
7,| 2 | TABLE ACCESS FULL| DEPARTMENTS | 1 | 3 (0)|
8,| 3 | TABLE ACCESS FULL| EMPLOYEES | 30 | 3 (0)|
9,---------------------------------------------------------------


**Why / traps:** Read the plan from the **most indented** line up. `TABLE ACCESS FULL` = read the whole table. `INDEX UNIQUE SCAN` = use the primary key index. `HASH JOIN` / `NESTED LOOPS` = join methods. Rows and cost are the optimizer's **estimates**.

## Q7. Index and statistics  ·  🟡 medium

Copy orders to `prac_orders`, create an **index** on customer_id, gather **statistics**, then explain `SELECT * FROM prac_orders WHERE customer_id = 7`.

**How it works:** Copy the table, add an index, gather statistics, then explain the query and see if the plan uses the index.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_orders PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_orders AS SELECT * FROM orders

OK, 108 row(s) affected

In [ ]:
%%oracle
CREATE INDEX prac_orders_cust_ix ON prac_orders (customer_id)

OK

In [ ]:
%%oracle
BEGIN DBMS_STATS.GATHER_TABLE_STATS(USER, 'PRAC_ORDERS'); END;

OK

In [ ]:
%%oracle
EXPLAIN PLAN FOR SELECT * FROM prac_orders WHERE customer_id = 7

OK

In [ ]:
%%oracle
SELECT plan_table_output FROM TABLE(DBMS_XPLAN.DISPLAY(format => 'BASIC +ROWS'))

,plan_table_output
0,Plan hash value: 443059126
1,
2,---------------------------------------------------------------------------
3,| Id | Operation | Name | Rows |
4,---------------------------------------------------------------------------
5,| 0 | SELECT STATEMENT | | 3 |
6,| 1 | TABLE ACCESS BY INDEX ROWID BATCHED| PRAC_ORDERS | 3 |
7,| 2 | INDEX RANGE SCAN | PRAC_ORDERS_CUST_IX | 3 |
8,---------------------------------------------------------------------------


**Why / traps:** **Statistics** (row counts, value spread) let the optimizer guess well. An index helps when a query returns **few** rows. On a tiny table a full scan can still be cheaper. An index is not used when you wrap the column in a function (e.g. `TRUNC(order_date)`), unless you create a function-based index.

## Q8. Oracle partitioned table  ·  🔴 hard

Create `prac_orders_part` with **interval partitioning** by month on order_date, from the clean orders. Count its partitions, then explain a query for October 2025 only.

**How it works:** Interval partitioning makes one partition per month; the plan for a one-month filter shows only that partition is read.

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_orders_part PURGE

OK

In [ ]:
%%oracle
CREATE TABLE prac_orders_part
PARTITION BY RANGE (order_date) INTERVAL (NUMTOYMINTERVAL(1, 'MONTH'))
  (PARTITION p_old VALUES LESS THAN (DATE '2023-01-01'))
AS SELECT DISTINCT * FROM orders

OK

In [ ]:
%%oracle
SELECT COUNT(*) AS num_partitions FROM user_tab_partitions WHERE table_name = 'PRAC_ORDERS_PART'

,num_partitions
0,30


In [ ]:
%%oracle
EXPLAIN PLAN FOR
SELECT * FROM prac_orders_part
WHERE order_date >= DATE '2025-10-01' AND order_date < DATE '2025-11-01'

OK

In [ ]:
%%oracle
SELECT plan_table_output FROM TABLE(DBMS_XPLAN.DISPLAY(format => 'BASIC +PARTITION'))

,plan_table_output
0,Plan hash value: 1674664728
1,
2,-------------------------------------------------------------------
3,| Id | Operation | Name | Pstart| Pstop |
4,-------------------------------------------------------------------
5,| 0 | SELECT STATEMENT | | | |
6,| 1 | PARTITION RANGE SINGLE| | 35 | 35 |
7,| 2 | TABLE ACCESS FULL | PRAC_ORDERS_PART | 35 | 35 |
8,-------------------------------------------------------------------


**Why / traps:** **Interval partitioning** creates a new monthly partition by itself when data arrives. `PARTITION RANGE SINGLE` with Pstart = Pstop means only one partition is read (pruning).